# 01 – Sanity-check EDA for IPL CricSheet data

This notebook verifies the output of `src/parse_cricsheet.py` before moving to Step 2.

In [ ]:
import pandas as pd

matches = pd.read_parquet('../data/processed/matches.parquet')
balls = pd.read_parquet('../data/processed/balls.parquet')

print('Matches shape:', matches.shape)
print('Balls shape  :', balls.shape)

## 1. Match count by season (should show 2008 → 2025)

In [ ]:
matches['season'].value_counts().sort_index()

## 2. Any matches missing ball-by-ball data (set diff)

In [ ]:
match_ids_with_balls = set(balls['match_id'].unique())
match_ids_all = set(matches['match_id'].unique())
missing = match_ids_all - match_ids_with_balls
print(f'Matches without ball data: {len(missing)}')
if missing:
    print(sorted(missing))

## 3. Innings total distribution (runs_total summed by match+innings)

Max should be roughly 260, mean ~160.

In [ ]:
innings_totals = balls.groupby(['match_id', 'innings'])['runs_total'].sum().reset_index(name='innings_total')
print(innings_totals['innings_total'].describe())

## 4. Result value counts (how many no-results and ties)

In [ ]:
matches['result'].value_counts(dropna=False)

## 5. Top 10 most sixes by a batter in a single match

The top row should plausibly be Chris Gayle ~17 sixes in 2013.

In [ ]:
sixes = balls[balls['runs_batter'] == 6]
top_sixes = (
    sixes.groupby(['match_id', 'batter'])
    .size()
    .reset_index(name='sixes')
    .sort_values('sixes', ascending=False)
    .head(10)
    .merge(matches[['match_id', 'date', 'season']], on='match_id', how='left')
)
top_sixes